# Medicare Telehealth Fraud and Utilization Analytics**Version:** REBUILD V3**Author:** Seemab Hassan**Date:** May 14, 2026## Methodology framingThis notebook implements anomaly detection and risk stratification on Medicare telehealth billing data. It is **not** a supervised fraud classifier. The system produces anomaly scores indicating statistical deviation from peer billing patterns. Flagged providers warrant investigative review by appropriate authorities; the system does not make fraud determinations.## Data sources- Medicare Physician and Other Practitioners by Provider and Service files 2019 to 2023 (CMS public use)- OIG LEIE Exclusion Database- CMS TMEDTREND Public file (telehealth utilization context)- HHS-OIG and DOJ telehealth program integrity sources## Validation strategy: six pillars1. Bootstrap confidence intervals on headline metrics (Section 10)2. Permutation feature importance for the Isolation Forest (Section 11)3. Temporal holdout validation, train on 2021 plus 2022, hold out 2023 (Section 12)4. Honest OIG LEIE concordance reporting (Section 13)5. DOJ defendant cross reference (Section 14)6. Telehealth fraud typology pattern alignment (Section 15)7. Threshold sensitivity analysis (Section 16)

## Section 1: Setup and Configuration

In [ ]:
import os, sys, json, time, hashlib, platformfrom pathlib import Pathfrom datetime import datetime, timezoneimport numpy as npimport pandas as pdfrom scipy import statsfrom scipy.stats import spearmanrfrom sklearn.ensemble import IsolationForestfrom sklearn.preprocessing import StandardScalerfrom difflib import SequenceMatcherimport matplotlib.pyplot as pltimport seaborn as snsnp.random.seed(42)print(f"Python {sys.version.split()[0]}")print(f"numpy {np.__version__}, pandas {pd.__version__}")import sklearn, scipyprint(f"scikit-learn {sklearn.__version__}, scipy {scipy.__version__}")

In [ ]:
# Path configuration (edit BASE_PATH if your dataset folder lives elsewhere)BASE_PATH = Path(r"D:\\DATA_SETS\\DATA_SET_TELEHEALTH FRAUD DETECTION SYSTEM")DATASETS = BASE_PATH / "Datasets"PATHS = {    "provider_service_2019": BASE_PATH / "Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2019" / "Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2019.csv",    "provider_service_2020": BASE_PATH / "Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2020" / "Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2020.csv",    "provider_service_2021": BASE_PATH / "Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2021" / "Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2021.csv",    "provider_service_2022": BASE_PATH / "Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2022" / "Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2022.csv",    "provider_service_2023": BASE_PATH / "Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2023" / "Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2023.csv",    "leie": DATASETS / "LEIE Database.csv",    "tmedtrend": DATASETS / "TMEDTREND_PUBLIC_250522.csv",    "taxonomy": DATASETS / "Medicare_Provider_and_Supplier_Taxonomy_Crosswalk_JAN__2025.csv",    "govt_source": Path(r"D:\\Seemab_FInal_Documents\\Project 5_Telehealth Fraud and Utilization Analytics\\GOVT_DOCUMENTS\\GOVT_SOURCE_PROJECT_5.pdf"),    "news_source": Path(r"D:\\Seemab_FInal_Documents\\Project 5_Telehealth Fraud and Utilization Analytics\\NEWS_SOURCES\\News_Article_Project_5.pdf"),}REBUILD = Path(r"D:\\Seemab_FInal_Documents\\Project 5_Telehealth Fraud and Utilization Analytics\\REBUILD")RESULTS = REBUILD / "results"FIGURES = REBUILD / "figures"FILTERED = RESULTS / "filtered"for p in [RESULTS, FIGURES, FILTERED]:    p.mkdir(parents=True, exist_ok=True)print("Paths configured.")

In [ ]:
# File inventory with SHA-256 hashesdef sha256_file(path, chunk_size=4*1024*1024):    h = hashlib.sha256()    with open(path, "rb") as f:        while True:            data = f.read(chunk_size)            if not data: break            h.update(data)    return h.hexdigest()inventory = []for key, p in PATHS.items():    if not p.exists():        inventory.append({"key": key, "exists": False, "path": str(p)})        continue    sz = os.path.getsize(p)    if sz < 500 * 1024 * 1024:  # hash small files now        sha = sha256_file(p)    else:        sha = "(skipped; large file - precomputed in REBUILD_LOG.md)"    inventory.append({"key": key, "exists": True, "size_bytes": sz, "size_gb": round(sz/1e9, 3), "sha256": sha})inv = pd.DataFrame(inventory)print(inv.to_string(index=False))

## Section 2: Data Loading with Telehealth HCPCS Filter

**Telehealth HCPCS code basket (27 codes):**- Online digital E/M: 99421, 99422, 99423- Telephone E/M: 99441, 99442, 99443- Remote physiologic monitoring: 99453, 99454, 99457, 99458, 99091- Brief virtual check-ins and originating site: G2010, G2012, G2061, G2062, G2063, G2066, G2250, G2251, G2252, Q3014- Communication tech-based by qualified nonphysician: 98966, 98967, 98968, 98970, 98971, 98972**Filter rules:** keep rows where HCPCS_Cd is in basket AND Tot_Benes >= 11 (CMS suppression threshold) AND Avg_Mdcr_Pymt_Amt > 0.

In [ ]:
TELEHEALTH_CODES = {    "99421","99422","99423","99441","99442","99443",    "99453","99454","99457","99458","99091",    "G2010","G2012","G2061","G2062","G2063","G2066",    "G2250","G2251","G2252","Q3014",    "98966","98967","98968","98970","98971","98972"}AUDIO_ONLY = {"99441","99442","99443","98966","98967","98968"}USECOLS = ["Rndrng_NPI","Rndrng_Prvdr_Last_Org_Name","Rndrng_Prvdr_First_Name","Rndrng_Prvdr_City","Rndrng_Prvdr_State_Abrvtn","Rndrng_Prvdr_Zip5","Rndrng_Prvdr_RUCA","Rndrng_Prvdr_Type","HCPCS_Cd","HCPCS_Desc","Place_Of_Srvc","Tot_Benes","Tot_Srvcs","Tot_Bene_Day_Srvcs","Avg_Sbmtd_Chrg","Avg_Mdcr_Alowd_Amt","Avg_Mdcr_Pymt_Amt","Avg_Mdcr_Stdzd_Amt"]DTYPE = {"HCPCS_Cd": str, "Rndrng_NPI": str, "Rndrng_Prvdr_RUCA": str}def filter_year(year, force=False):    out = FILTERED / f"telehealth_{year}.csv"    if out.exists() and not force:        print(f"  {year}: cache exists, skipping ({out})")        return pd.read_csv(out, dtype=DTYPE)    t0 = time.time()    parts = []    for ch in pd.read_csv(PATHS[f"provider_service_{year}"], usecols=USECOLS, dtype=DTYPE, chunksize=2_000_000, engine='c'):        m = ch["HCPCS_Cd"].isin(TELEHEALTH_CODES)        parts.append(ch.loc[m])    df = pd.concat(parts, ignore_index=True)    df.to_csv(out, index=False)    print(f"  {year}: {len(df):,} rows filtered ({time.time()-t0:.1f}s)")    return dffiltered = {y: filter_year(y) for y in [2019, 2020, 2021, 2022, 2023]}for y, df in filtered.items():    print(f"{y}: {len(df):,} telehealth rows, {df['Rndrng_NPI'].nunique():,} unique NPIs")

## Section 3: Feature Engineering

**Feature formulas (per provider after aggregating telehealth HCPCS rows):**- `Telehealth_Services_Total` = sum of Tot_Srvcs across telehealth rows- `Telehealth_Benes_Max` = max of Tot_Benes (unique beneficiary proxy)- `Telehealth_Payment_Total` = sum of (Avg_Mdcr_Pymt_Amt * Tot_Srvcs)- `Telehealth_HCPCS_Diversity` = count of distinct HCPCS codes- `Audio_Only_Services` = sum of Tot_Srvcs for codes in audio only basket- `Services_Per_Bene` = Telehealth_Services_Total / Telehealth_Benes_Max- `Payment_Per_Bene` = Telehealth_Payment_Total / Telehealth_Benes_Max- `Avg_Payment_Per_Service` = Telehealth_Payment_Total / Telehealth_Services_Total- `Audio_Only_Share` = Audio_Only_Services / Telehealth_Services_Total- `Log_Services` = log(1 + Telehealth_Services_Total) for scale normalization

In [ ]:
def build_universe(df):    df = df[(df["Tot_Benes"] >= 11) & (df["Avg_Mdcr_Pymt_Amt"] > 0)].copy()    df["Row_Payment"] = df["Avg_Mdcr_Pymt_Amt"] * df["Tot_Srvcs"]    df["Is_Audio_Only"] = df["HCPCS_Cd"].isin(AUDIO_ONLY).astype(int)    agg = df.groupby("Rndrng_NPI").agg(        Telehealth_Services_Total=("Tot_Srvcs", "sum"),        Telehealth_Benes_Max=("Tot_Benes", "max"),        Telehealth_Payment_Total=("Row_Payment", "sum"),        Telehealth_HCPCS_Diversity=("HCPCS_Cd", "nunique"),        Audio_Only_Services=("Is_Audio_Only", lambda x: (df.loc[x.index, "Tot_Srvcs"] * x).sum()),    ).reset_index()    meta = df.groupby("Rndrng_NPI").agg(        Last_Name=("Rndrng_Prvdr_Last_Org_Name", "first"),        First_Name=("Rndrng_Prvdr_First_Name", "first"),        City=("Rndrng_Prvdr_City", "first"),        State=("Rndrng_Prvdr_State_Abrvtn", "first"),        Zip5=("Rndrng_Prvdr_Zip5", "first"),        RUCA=("Rndrng_Prvdr_RUCA", "first"),        Provider_Type=("Rndrng_Prvdr_Type", "first"),    ).reset_index()    u = meta.merge(agg, on="Rndrng_NPI")    u["Services_Per_Bene"] = u["Telehealth_Services_Total"] / u["Telehealth_Benes_Max"].replace(0, np.nan)    u["Payment_Per_Bene"] = u["Telehealth_Payment_Total"] / u["Telehealth_Benes_Max"].replace(0, np.nan)    u["Avg_Payment_Per_Service"] = u["Telehealth_Payment_Total"] / u["Telehealth_Services_Total"].replace(0, np.nan)    u["Audio_Only_Share"] = u["Audio_Only_Services"] / u["Telehealth_Services_Total"].replace(0, np.nan)    u["Log_Services"] = np.log1p(u["Telehealth_Services_Total"])    for col in ["Services_Per_Bene","Payment_Per_Bene","Avg_Payment_Per_Service","Audio_Only_Share"]:        u[col] = u[col].replace([np.inf, -np.inf], np.nan).fillna(u[col].median())    return uuniverse_2023 = build_universe(filtered[2023])print(f"2023 analytical universe: {len(universe_2023):,} unique providers")print(f"Total telehealth services: {universe_2023['Telehealth_Services_Total'].sum():,.0f}")print(f"Total telehealth payments: ${universe_2023['Telehealth_Payment_Total'].sum():,.2f}")print(f"States represented: {universe_2023['State'].nunique()}")print(f"Provider types: {universe_2023['Provider_Type'].nunique()}")

## Section 4: Peer Baselines (state level)

In [ ]:
state_baselines = universe_2023.groupby("State").agg(    State_Median_PaymentPerBene=("Payment_Per_Bene","median"),    State_Median_ServicesPerBene=("Services_Per_Bene","median"),    State_Provider_Count=("Rndrng_NPI","count")).reset_index()state_baselines.to_csv(RESULTS / "state_baselines.csv", index=False)universe_2023 = universe_2023.merge(state_baselines[["State","State_Median_PaymentPerBene","State_Median_ServicesPerBene"]], on="State", how="left")print(state_baselines.head(10).to_string(index=False))

## Section 5: Statistical Risk Layer

**Documented per-feature z-score weights (sum to 1.0):**| Feature | Weight | Rationale ||---|---|---|| Payment_Per_Bene | 0.30 | Primary signal: excess payment per beneficiary || Services_Per_Bene | 0.25 | Service intensity per beneficiary || Avg_Payment_Per_Service | 0.15 | Per-service price anomaly || Audio_Only_Share | 0.10 | Audio-only concentration risk || Telehealth_HCPCS_Diversity | 0.10 | Code diversity (low diversity = single-code dominance) || Log_Services | 0.10 | Scale-normalized volume |

In [ ]:
WEIGHTS = {"Payment_Per_Bene":0.30,"Services_Per_Bene":0.25,"Avg_Payment_Per_Service":0.15,"Audio_Only_Share":0.10,"Telehealth_HCPCS_Diversity":0.10,"Log_Services":0.10}assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9stat_raw = np.zeros(len(universe_2023))for f, w in WEIGHTS.items():    mu, sd = universe_2023[f].mean(), universe_2023[f].std()    if sd > 0:        stat_raw += w * ((universe_2023[f] - mu) / sd).abs()universe_2023["Statistical_Risk_Raw"] = stat_rawprint(f"Statistical risk distribution: mean={stat_raw.mean():.3f}, median={np.median(stat_raw):.3f}, max={stat_raw.max():.3f}")

## Section 6: Machine Learning Risk Layer (Isolation Forest)

In [ ]:
ML_FEATS = ["Payment_Per_Bene","Services_Per_Bene","Avg_Payment_Per_Service","Audio_Only_Share","Telehealth_HCPCS_Diversity","Log_Services","Telehealth_Benes_Max","Telehealth_Services_Total"]X = universe_2023[ML_FEATS].fillna(universe_2023[ML_FEATS].median()).valuesscaler = StandardScaler()X_scaled = scaler.fit_transform(X)# Sanity checkassert abs(X_scaled.mean()) < 0.01 and abs(X_scaled.std() - 1) < 0.01iso = IsolationForest(contamination=0.02, n_estimators=100, random_state=42, bootstrap=False)iso.fit(X_scaled)# Invert so higher = more anomalousuniverse_2023["ML_Risk_Raw"] = -iso.score_samples(X_scaled)print(f"ML risk: mean={universe_2023['ML_Risk_Raw'].mean():.4f}, median={universe_2023['ML_Risk_Raw'].median():.4f}, max={universe_2023['ML_Risk_Raw'].max():.4f}")

## Section 7: Combined Risk Score and Tier Stratification

In [ ]:
def minmax(s):    lo, hi = s.min(), s.max()    return (s - lo) / (hi - lo) if hi > lo else s * 0universe_2023["Stat_Norm"] = minmax(universe_2023["Statistical_Risk_Raw"])universe_2023["ML_Norm"] = minmax(universe_2023["ML_Risk_Raw"])universe_2023["Combined_Risk_Score"] = (0.60 * universe_2023["Stat_Norm"] + 0.40 * universe_2023["ML_Norm"]) * 100universe_2023["Percentile"] = universe_2023["Combined_Risk_Score"].rank(pct=True) * 100def tier(p):    if p < 80: return "Low"    if p < 95: return "Medium"    if p < 99: return "High"    if p < 99.9: return "Critical"    return "Extreme"universe_2023["Tier"] = universe_2023["Percentile"].apply(tier)tier_counts = universe_2023["Tier"].value_counts().reindex(["Low","Medium","High","Critical","Extreme"]).fillna(0).astype(int)tier_counts.to_csv(RESULTS / "tier_distribution.csv")print("Tier distribution:")print(tier_counts)flagged = universe_2023[universe_2023["Percentile"] >= 95].copy().sort_values("Combined_Risk_Score", ascending=False)print(f"\nFlagged (top 5%): {len(flagged):,}")universe_2023.to_csv(RESULTS / "scored_universe_2023.csv", index=False)

## Section 8: Per-Provider Excess Billing and Per-State Aggregation

In [ ]:
flagged["Excess_Payment_Per_Bene"] = (flagged["Payment_Per_Bene"] - flagged["State_Median_PaymentPerBene"]).clip(lower=0)flagged["Excess_Billing"] = flagged["Excess_Payment_Per_Bene"] * flagged["Telehealth_Benes_Max"]flagged_sorted = flagged.sort_values("Excess_Billing", ascending=False)flagged_sorted.to_csv(RESULTS / "flagged_providers_2023.csv", index=False)sample_excess = flagged["Excess_Billing"].sum()print(f"Sample excess billing: ${sample_excess:,.2f}")print("\nTop 10 flagged by excess billing:")print(flagged_sorted.head(10)[["Rndrng_NPI","Last_Name","First_Name","State","Provider_Type","Telehealth_Services_Total","Telehealth_Benes_Max","Payment_Per_Bene","Combined_Risk_Score","Excess_Billing"]].to_string(index=False))state_agg = flagged.groupby("State").agg(    Flagged_Providers=("Rndrng_NPI","count"),    Total_Excess=("Excess_Billing","sum"),    Total_Payment=("Telehealth_Payment_Total","sum"),    Beneficiaries_Served=("Telehealth_Benes_Max","sum")).reset_index().sort_values("Total_Excess", ascending=False)state_agg["Excess_Millions"] = state_agg["Total_Excess"]/1e6state_agg.to_csv(RESULTS / "per_state_excess_2023.csv", index=False)print("\nTop 10 states:")print(state_agg.head(10).to_string(index=False))

## Section 9: Transparent National Projection

Coverage multiplier derived from CMS TMEDTREND public file. **No recovery rate multiplier. No enforcement effectiveness factor.** Reports both sample figure and projected figure separately.

In [ ]:
tmed = pd.read_csv(PATHS["tmedtrend"], dtype=str)tmed_overall = tmed[(tmed["quarter"]=="Overall") & (tmed["Bene_Geo_Desc"]=="National") &                     (tmed["Bene_Mdcd_Mdcr_Enrl_Stus"]=="All") & (tmed["Bene_Race_Desc"]=="All") &                    (tmed["Bene_Sex_Desc"]=="All") & (tmed["Bene_Mdcr_Entlmt_Stus"]=="All") &                    (tmed["Bene_Age_Desc"]=="All") & (tmed["Bene_RUCA_Desc"]=="All")].copy()tmed_overall["Year"] = tmed_overall["Year"].astype(int)for c in ["Total_Bene_Telehealth","Total_PartB_Enrl","Pct_Telehealth"]:    tmed_overall[c] = pd.to_numeric(tmed_overall[c])print("National telehealth utilization by year:")print(tmed_overall[["Year","Total_Bene_Telehealth","Total_PartB_Enrl","Pct_Telehealth"]].to_string(index=False))tmed_overall.to_csv(RESULTS / "tmedtrend_national.csv", index=False)nat_2023 = tmed_overall[tmed_overall["Year"]==2023].iloc[0]universe_benes = universe_2023["Telehealth_Benes_Max"].sum()coverage_mult = float(nat_2023["Total_Bene_Telehealth"]) / float(universe_benes)projected = sample_excess * coverage_multprint(f"\n2023 national telehealth benes: {nat_2023['Total_Bene_Telehealth']:,.0f}")print(f"Universe Telehealth_Benes_Max sum: {universe_benes:,.0f}")print(f"Coverage multiplier: {coverage_mult:.3f}")print(f"Sample excess: ${sample_excess:,.2f}")print(f"Projected national excess (coverage-adjusted): ${projected:,.2f}")with open(RESULTS / "projection_methodology.json","w") as f:    json.dump({"sample_excess_usd": float(sample_excess), "coverage_multiplier": coverage_mult, "projected_national_excess_usd": float(projected), "methodology": "Inverse coverage multiplier. No recovery rate. No enforcement factor."}, f, indent=2)

## Section 10: Bootstrap Confidence Intervals (n=1000)

In [ ]:
n_iter = 1000scores = universe_2023["Combined_Risk_Score"].valuesppb = universe_2023["Payment_Per_Bene"].valuesbenes = universe_2023["Telehealth_Benes_Max"].valuesstate = universe_2023["State"].valuesunique_states = np.unique(state)state_idx = {s:i for i,s in enumerate(unique_states)}state_codes = np.array([state_idx[s] for s in state])flagged_counts = np.zeros(n_iter)sample_excesses = np.zeros(n_iter)rng = np.random.RandomState(42)n = len(universe_2023)for it in range(n_iter):    idx = rng.randint(0, n, size=n)    s_scores = scores[idx]; s_ppb = ppb[idx]; s_benes = benes[idx]; s_state = state_codes[idx]    cutoff = np.percentile(s_scores, 95)    mask = s_scores >= cutoff    flagged_counts[it] = mask.sum()    medians = np.full(len(unique_states), np.nan)    for si in range(len(unique_states)):        m = s_state == si        if m.any():            medians[si] = np.median(s_ppb[m])    med_per_row = medians[s_state]    sample_excesses[it] = (np.maximum(0, s_ppb - med_per_row) * s_benes)[mask].sum()bs = pd.DataFrame({"iteration": range(n_iter), "flagged_count": flagged_counts.astype(int), "sample_excess": sample_excesses})bs.to_csv(RESULTS / "bootstrap.csv", index=False)print(f"Bootstrap (n=1000) sample excess: mean=${sample_excesses.mean():,.0f}")print(f"  95% CI: [${np.percentile(sample_excesses,2.5):,.0f}, ${np.percentile(sample_excesses,97.5):,.0f}]")print(f"Flagged count 95% CI: [{np.percentile(flagged_counts,2.5):.0f}, {np.percentile(flagged_counts,97.5):.0f}]")

## Section 11: Permutation Feature Importance

Replaces the fabricated Isolation Forest feature importance table from the original notebook. For each ML feature, shuffle the column, recompute anomaly scores, measure drop in Spearman rank correlation with original scores. Average across 5 repeats.

In [ ]:
base_scores = -iso.score_samples(X_scaled)n_repeats = 5importances = []rng = np.random.RandomState(42)for i, feat in enumerate(ML_FEATS):    drops = []    for r in range(n_repeats):        Xp = X_scaled.copy()        rng.shuffle(Xp[:, i])        perm_scores = -iso.score_samples(Xp)        rho, _ = spearmanr(base_scores, perm_scores)        drops.append(1 - rho)    importances.append(np.mean(drops))imp = np.array(importances)imp_norm = imp / imp.sum() * 100fi = pd.DataFrame({"Feature": ML_FEATS, "Importance_Raw": imp, "Importance_Pct": imp_norm}).sort_values("Importance_Pct", ascending=False)fi.to_csv(RESULTS / "permutation_importance.csv", index=False)print("Permutation feature importance:")print(fi.to_string(index=False))

## Section 12: Temporal Holdout Validation

Train on 2021 plus 2022 (concatenated telehealth universe), hold out 2023. Identify training-flagged NPIs (top 5 percent), locate in holdout, measure lift versus population.

In [ ]:
u2021 = build_universe(filtered[2021])u2022 = build_universe(filtered[2022])train = pd.concat([u2021, u2022], ignore_index=True)train_agg = train.groupby("Rndrng_NPI").agg({c:"mean" for c in ["Telehealth_Services_Total","Telehealth_Benes_Max","Telehealth_Payment_Total","Telehealth_HCPCS_Diversity","Audio_Only_Services","Services_Per_Bene","Payment_Per_Bene","Avg_Payment_Per_Service","Audio_Only_Share","Log_Services"]}).reset_index()print(f"Training pool: {len(train_agg):,} unique NPIs")# Re-run the scoring pipeline on the training pooldef score_pool(df):    s = np.zeros(len(df))    for f, w in WEIGHTS.items():        mu, sd = df[f].mean(), df[f].std()        if sd > 0:            s += w * ((df[f] - mu) / sd).abs()    df["Stat"] = s    Xt = df[ML_FEATS].fillna(df[ML_FEATS].median()).values    Xts = StandardScaler().fit_transform(Xt)    iso2 = IsolationForest(contamination=0.02, n_estimators=100, random_state=42, bootstrap=False)    iso2.fit(Xts)    df["ML"] = -iso2.score_samples(Xts)    df["Combined"] = (0.60*minmax(df["Stat"]) + 0.40*minmax(df["ML"]))*100    df["Pct"] = df["Combined"].rank(pct=True)*100    return dftrain_scored = score_pool(train_agg.copy())train_flagged = set(train_scored[train_scored["Pct"]>=95]["Rndrng_NPI"])holdout = universe_2023[universe_2023["Rndrng_NPI"].isin(train_flagged)]coverage = len(holdout) / len(train_flagged) * 100pop_mean = universe_2023["Payment_Per_Bene"].mean()pop_median = universe_2023["Payment_Per_Bene"].median()flagged_mean = holdout["Payment_Per_Bene"].mean()flagged_median = holdout["Payment_Per_Bene"].median()print(f"Training-flagged NPIs: {len(train_flagged):,}")print(f"Reappear in 2023 holdout: {len(holdout):,} ({coverage:.1f}%)")print(f"Mean lift: {flagged_mean / pop_mean:.2f}x (flagged ${flagged_mean:.2f} vs pop ${pop_mean:.2f})")print(f"Median lift: {flagged_median / pop_median:.2f}x")th = pd.DataFrame([{"training_pool_npis": len(train_agg), "training_flagged_npis": len(train_flagged), "holdout_coverage_pct": coverage, "mean_lift_ratio": flagged_mean/pop_mean, "median_lift_ratio": flagged_median/pop_median, "population_mean_payment_per_bene": pop_mean, "flagged_carryover_mean_payment_per_bene": flagged_mean}])th.to_csv(RESULTS / "temporal_holdout.csv", index=False)

## Section 13: Honest OIG LEIE Concordance

Direct NPI matching plus fuzzy name matching at 0.85 SequenceMatcher threshold. **Framing:** LEIE concordance is a name-pattern surveillance signal, not a fraud probability validation. Excluded providers are by definition barred from billing Medicare; direct matches in active billing files indicate either reinstatement-status errors or fraudulent continued billing.

In [ ]:
leie = pd.read_csv(PATHS["leie"], dtype=str)leie["NPI"] = leie["NPI"].fillna("").astype(str).str.strip()leie_npis = set(leie.loc[leie["NPI"].str.match(r"^\d{10}$"), "NPI"])print(f"LEIE excluded entities: {len(leie):,}, with valid NPI: {len(leie_npis):,}")flagged_sorted["LEIE_Direct_Match"] = flagged_sorted["Rndrng_NPI"].isin(leie_npis)direct = flagged_sorted[flagged_sorted["LEIE_Direct_Match"]]print(f"\nDirect NPI matches in flagged set: {len(direct)}")if len(direct) > 0:    print(direct[["Rndrng_NPI","Last_Name","First_Name","State","Provider_Type","Combined_Risk_Score","Excess_Billing"]].to_string(index=False))# Fuzzy name match: top 100 by excessdef best_match(name, candidates, threshold=0.85):    name_up = str(name).upper().strip()    if not name_up: return None, 0    prefix = name_up[:3] if len(name_up) >= 3 else name_up    cands = [c for c in candidates if c.startswith(prefix)]    best = (None, 0)    for c in cands:        r = SequenceMatcher(None, name_up, c).ratio()        if r > best[1]: best = (c, r)    return (best[0], best[1]) if best[1] >= threshold else (None, best[1])leie["BUSNAME"] = leie["BUSNAME"].fillna("").str.upper().str.strip()leie["LASTNAME"] = leie["LASTNAME"].fillna("").str.upper().str.strip()leie["FIRSTNAME"] = leie["FIRSTNAME"].fillna("").str.upper().str.strip()indnames = list((leie["LASTNAME"] + " " + leie["FIRSTNAME"]).str.strip().unique())busnames = list(leie["BUSNAME"][leie["BUSNAME"]!=""].unique())top100 = flagged_sorted.head(100)matches = []for _, row in top100.iterrows():    ln = str(row.get("Last_Name","")).upper().strip()    fn = str(row.get("First_Name","")).upper().strip()    full = (ln + " " + fn).strip() if fn and fn != "NAN" else ln    candidates = busnames if (fn=="" or fn=="NAN") else indnames    m, score = best_match(full, candidates)    if m:        matches.append({"NPI": row["Rndrng_NPI"], "Provider": full, "LEIE_Match": m, "Similarity": score})fm = pd.DataFrame(matches)fm.to_csv(RESULTS / "leie_concordance.csv", index=False)print(f"\nFuzzy LEIE matches (top 100 flagged, 0.85 threshold): {len(fm)}")if len(fm) > 0:    print(fm.to_string(index=False))

## Section 14: DOJ Prosecution Cross Reference

**Framing:** DOJ named defendants are typically prosecuted because their schemes have already been discovered. By the time names appear in public press releases, the providers' NPIs are typically suspended or excluded. Matches in active billing files are not expected. A zero match count is the expected result and validates the framing.

In [ ]:
try:    import pdfplumber    def extract_pdf_text(p):        text = ""        with pdfplumber.open(p) as pdf:            for pg in pdf.pages:                t = pg.extract_text()                if t: text += t + "\n"        return text    news = extract_pdf_text(PATHS["news_source"]) if PATHS["news_source"].exists() else ""    govt = extract_pdf_text(PATHS["govt_source"]) if PATHS["govt_source"].exists() else ""    combined = news + "\n" + govt    print(f"NEWS PDF: {len(news):,} chars; GOVT PDF: {len(govt):,} chars")except ImportError:    print("pdfplumber not installed. Run: pip install pdfplumber")    combined = ""if combined:    import re    person_pat = r'\b[A-Z][a-z]+(?:\s+[A-Z]\.?)?\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+)?\b'    persons = set(re.findall(person_pat, combined))    stop = {"United States","Department Justice","Department Health"}    persons = {p for p in persons if p not in stop and not p.startswith(("United","Department","Office"))}    print(f"Extracted {len(persons)} candidate names")    doj_matches = []    for _, row in flagged_sorted.head(500).iterrows():        full = (str(row.get("Last_Name","")) + " " + str(row.get("First_Name",""))).strip().upper()        if not full or len(full) < 5: continue        for p in persons:            r = SequenceMatcher(None, full, p.upper()).ratio()            if r >= 0.85:                doj_matches.append({"NPI": row["Rndrng_NPI"], "Flagged_Name": full, "DOJ_Match": p, "Similarity": r})                break    dm = pd.DataFrame(doj_matches)    dm.to_csv(RESULTS / "doj_cross_reference.csv", index=False)    print(f"DOJ matches (top 500 flagged): {len(dm)}")

## Section 15: Telehealth Fraud Typology Pattern Alignment

Five telehealth-specific typologies:1. **Extreme payment per beneficiary:** Payment_Per_Bene > 10x state median2. **Audio only heavy:** Audio_Only_Share > 0.803. **Single HCPCS dominance:** Telehealth_HCPCS_Diversity <= 14. **Rural high volume:** Rural RUCA (8-10) AND >5,000 telehealth services5. **Pandemic emergence:** No 2019 telehealth presence AND >1,000 services in 2023

In [ ]:
typology = pd.DataFrame()typology["Rndrng_NPI"] = flagged_sorted["Rndrng_NPI"]state_med_map = universe_2023.groupby("State")["Payment_Per_Bene"].median().to_dict()state_med_per_row = flagged_sorted["State"].map(state_med_map).fillna(universe_2023["Payment_Per_Bene"].median())typology["T1_Extreme_PaymentPerBene"] = (flagged_sorted["Payment_Per_Bene"] > 10 * state_med_per_row).astype(int)typology["T2_Audio_Only_Heavy"] = (flagged_sorted["Audio_Only_Share"] > 0.8).astype(int)typology["T3_Single_HCPCS_Dominant"] = (flagged_sorted["Telehealth_HCPCS_Diversity"] <= 1).astype(int)ruca_num = pd.to_numeric(flagged_sorted["RUCA"], errors="coerce")typology["T4_Rural_HighVolume"] = ((ruca_num >= 7.5) & (flagged_sorted["Telehealth_Services_Total"] > 5000)).astype(int)npis_2019 = set(filtered[2019]["Rndrng_NPI"])typology["T5_Pandemic_Emergence"] = ((~flagged_sorted["Rndrng_NPI"].isin(npis_2019)) & (flagged_sorted["Telehealth_Services_Total"] > 1000)).astype(int)typology["Total_Typology_Hits"] = typology[[c for c in typology.columns if c.startswith("T")]].sum(axis=1)typology.to_csv(RESULTS / "fraud_typology.csv", index=False)print(f"Telehealth fraud typology counts (flagged set, n={len(typology)}):")for c in ["T1_Extreme_PaymentPerBene","T2_Audio_Only_Heavy","T3_Single_HCPCS_Dominant","T4_Rural_HighVolume","T5_Pandemic_Emergence"]:    n = typology[c].sum()    print(f"  {c}: {n:,} ({n/len(typology)*100:.1f}%)")print(f"\nMulti-typology (2+ patterns): {(typology['Total_Typology_Hits']>=2).sum():,}")

## Section 16: Threshold Sensitivity

In [ ]:
ths = [90, 92, 94, 95, 96, 97, 98, 99, 99.5, 99.9]rows = []sm_full = universe_2023.groupby("State")["Payment_Per_Bene"].median().to_dict()for th_pct in ths:    cutoff = np.percentile(universe_2023["Combined_Risk_Score"], th_pct)    f = universe_2023[universe_2023["Combined_Risk_Score"] >= cutoff].copy()    f["State_Med"] = f["State"].map(sm_full)    f["Excess"] = (np.maximum(0, f["Payment_Per_Bene"] - f["State_Med"]) * f["Telehealth_Benes_Max"]).fillna(0)    rows.append({"Percentile": th_pct, "Flagged_Count": len(f), "Pct_of_Universe": len(f)/len(universe_2023)*100, "Sample_Excess": f["Excess"].sum(), "Mean_Excess_Per_Flagged": f["Excess"].sum()/len(f) if len(f)>0 else 0})ts = pd.DataFrame(rows)ts.to_csv(RESULTS / "threshold_sensitivity.csv", index=False)print(ts.to_string(index=False))

## Section 17: Reproducibility Envelope (RUN_INFO.json)

In [ ]:
run_info = {    "project": "Medicare Telehealth Fraud and Utilization Analytics",    "version": "REBUILD V3",    "rebuild_date": "2026-05-14",    "principal_investigator": "Seemab Hassan",    "methodology_framing": "Anomaly detection and risk stratification. NOT supervised fraud classification.",    "parameters": {        "random_seed": 42,        "isolation_forest_contamination": 0.02,        "isolation_forest_n_estimators": 100,        "statistical_weights": WEIGHTS,        "combined_weights": {"statistical": 0.60, "ml": 0.40},        "tier_cutpoints_percentile": {"Low":"<80","Medium":"80-95","High":"95-99","Critical":"99-99.9","Extreme":">=99.9"},        "flagged_set": "percentile >= 95",        "bootstrap_iterations": 1000,        "permutation_repeats": 5,        "fuzzy_threshold": 0.85    },    "telehealth_hcpcs_basket": sorted(TELEHEALTH_CODES),    "software": {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__, "scipy": scipy.__version__},    "source_file_hashes": {entry["key"]: entry.get("sha256","") for entry in inventory if entry.get("exists")},    "run_timestamp_utc": datetime.now(timezone.utc).isoformat()}with open(RESULTS / "RUN_INFO.json","w") as f:    json.dump(run_info, f, indent=2, default=str)print("RUN_INFO.json written.")

## Section 18: Unified Results Summary (single source of truth)

In [ ]:
summary = {    "headline": {        "active_telehealth_providers_2023": int(len(universe_2023)),        "total_telehealth_services_2023": int(universe_2023["Telehealth_Services_Total"].sum()),        "total_telehealth_payments_2023_usd": float(universe_2023["Telehealth_Payment_Total"].sum()),        "flagged_providers_top_5pct": int(len(flagged)),        "sample_excess_billing_usd": float(sample_excess),        "bootstrap_ci_low": float(np.percentile(sample_excesses,2.5)),        "bootstrap_ci_high": float(np.percentile(sample_excesses,97.5)),        "coverage_multiplier": coverage_mult,        "projected_national_excess_usd": float(projected)    },    "tier_distribution": tier_counts.to_dict(),    "top_state_by_excess": state_agg.iloc[0]["State"],    "permutation_top_feature": fi.iloc[0]["Feature"],    "temporal_holdout_coverage_pct": coverage,    "temporal_mean_lift": flagged_mean / pop_mean,    "leie_direct_matches": int(len(direct)),    "leie_fuzzy_top100": int(len(fm)) if 'fm' in dir() else 0,    "doj_matches": int(len(dm)) if 'dm' in dir() else 0,    "fraud_typology_pandemic_emergence_pct": float(typology["T5_Pandemic_Emergence"].sum() / len(typology) * 100),    "tmedtrend_2020_telehealth_share": float(tmed_overall[tmed_overall["Year"]==2020]["Pct_Telehealth"].iloc[0]),    "tmedtrend_2023_telehealth_share": float(tmed_overall[tmed_overall["Year"]==2023]["Pct_Telehealth"].iloc[0])}with open(RESULTS / "UNIFIED_RESULTS_SUMMARY.json","w") as f:    json.dump(summary, f, indent=2, default=str)print(json.dumps(summary, indent=2, default=str))